# Notebook 05 — Broadcasting and Vectorization

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 05 of 20  
**Prerequisites:** Notebook 04  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Broadcasting is the NumPy rule that makes `(5, 3) - (3,)` legal and fast.
Without it, every operation on arrays of different shapes would require an explicit
loop or manual reshaping. In practice, broadcasting shows up every time you
subtract per-sample means from a genes×samples matrix — exactly what z-scoring does.

---
## Step 2 — Intuition

The broadcasting rule: align shapes from the right. Where one array has a 1,
it is stretched to match the other. Where neither array has a 1 and the sizes
differ, the operation fails.

```
(5, 3)  - (3,)     → OK:  (3,) is treated as (1, 3) → stretched to (5, 3)
(5, 3)  - (5,)     → FAIL: (5,) aligns as (5,) which is ≠ (3,) on axis 1
(5, 3)  - (5, 1)   → OK:  (5, 1) stretched to (5, 3)
```

---
## Step 3 — Biological Background

**Library size normalization (CPM — counts per million):**

RNA-seq raw counts differ between samples because of sequencing depth, not biology.
To compare samples, each count is divided by the sample's total library size
and scaled to 1 million.

$$\text{CPM}_{ij} = \frac{X_{ij}}{\sum_i X_{ij}} \times 10^6$$

This is a broadcasting operation: divide each column by its column sum.
It cannot be done correctly without understanding which axis is which.

---
## Step 4 — Mathematical Explanation

Let $X \in \mathbb{R}^{m \times n}$ (genes × samples).  
Let $L = \mathbf{1}^T X \in \mathbb{R}^{1 \times n}$ (library sizes, one per sample).  
Then $\text{CPM} = (X / L) \times 10^6$.

In NumPy: `(X / X.sum(axis=0)) * 1e6` — but `X.sum(axis=0)` has shape `(n,)` and
`X` has shape `(m, n)`. Broadcasting promotes `(n,)` to `(1, n)` then stretches
to `(m, n)`. Each element in column $j$ is divided by the same value $L_j$.

---
## Step 5 — Computational Explanation

**When broadcasting saves time:**  
Broadcasting replaces explicit `for` loops. It is O(1) in the Python interpreter:
the stretching happens conceptually, not physically (no memory is allocated for
the stretched copy).

**Vectorization vs broadcasting:**
- *Vectorization*: applying an operation element-wise via a ufunc instead of a loop.
- *Broadcasting*: the shape-alignment rule that makes vectorized ops work between
  arrays of compatible but different shapes.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
rng = np.random.default_rng(seed=42)

# Synthetic raw count matrix: 6 genes, 4 samples
X = rng.negative_binomial(n=10, p=0.3, size=(6, 4)).astype(float)
gene_names = ["GeneA", "GeneB", "GeneC", "GeneD", "GeneE", "GeneF"]
print(f"Shape: {X.shape}")
print("Raw counts:")
print(X)

In [ ]:
# Cell 6.1 — CPM normalization using broadcasting
library_sizes = X.sum(axis=0)  # shape (4,)
print(f"Library sizes: {library_sizes}")
print(f"library_sizes.shape: {library_sizes.shape}")

# Broadcasting: X.shape=(6,4), library_sizes.shape=(4,) → (1,4) → (6,4)
CPM = X / library_sizes * 1e6
print(f"\nCPM matrix (shape {CPM.shape}):")
print(CPM.round(1))

# Verify: each column should sum to 1,000,000
print(f"\nColumn sums: {CPM.sum(axis=0).round(0)}")

In [ ]:
# Cell 6.2 — Broadcasting rule: shapes that work and those that don't
a = np.ones((5, 3))
b_ok_1 = np.ones((3,))    # promoted to (1,3) → (5,3): OK
b_ok_2 = np.ones((5, 1))  # stretched to (5,3): OK
b_fail = np.ones((5,))    # aligns as (5,) ≠ (3,) on axis 1: FAIL

print("(5,3) + (3,):",   (a + b_ok_1).shape)
print("(5,3) + (5,1):",  (a + b_ok_2).shape)

try:
    _ = a + b_fail
except ValueError as e:
    print(f"(5,3) + (5,): ValueError — {e}")

print("\nFix: reshape (5,) to (5,1) before operating:")
print("(5,3) + (5,1):", (a + b_fail.reshape(-1, 1)).shape)

In [ ]:
# Cell 6.3 — Vectorized pairwise distance (example: Euclidean distance between samples)
# Naive loop version:
import time

X_large = rng.exponential(5.0, size=(500, 20))
n_samples = X_large.shape[1]

def pairwise_dist_loop(X: np.ndarray) -> np.ndarray:
    n = X.shape[1]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            D[i, j] = D[j, i] = np.sqrt(((X[:, i] - X[:, j])**2).sum())
    return D

def pairwise_dist_vec(X: np.ndarray) -> np.ndarray:
    # X.T has shape (n_samples, n_genes)
    # Using broadcasting: (n, 1, p) - (1, n, p) → (n, n, p)
    Xt = X.T[:, np.newaxis, :]   # (n, 1, p)
    diff = Xt - X.T[np.newaxis, :, :]  # (n, n, p)
    return np.sqrt((diff**2).sum(axis=2))

t0 = time.perf_counter(); D_loop = pairwise_dist_loop(X_large); t_loop = time.perf_counter()-t0
t0 = time.perf_counter(); D_vec  = pairwise_dist_vec(X_large);  t_vec  = time.perf_counter()-t0

print(f"Loop:         {t_loop*1000:.2f} ms")
print(f"Vectorized:   {t_vec*1000:.2f} ms")
print(f"Speedup:      {t_loop/t_vec:.1f}×")
print(f"Results match: {np.allclose(D_loop, D_vec)}")

In [ ]:
# Cell 6.4 — np.einsum: the general contraction notation
# Euclidean distances using einsum — expressive and fast for linear algebra
def pairwise_dist_einsum(X: np.ndarray) -> np.ndarray:
    sq = np.einsum('ij,ij->j', X, X)  # sum of squares per column
    dots = X.T @ X                     # dot products (n_samples, n_samples)
    D2 = sq[:, None] - 2*dots + sq[None, :]  # broadcasting
    np.clip(D2, 0, None, out=D2)       # numerical stability
    return np.sqrt(D2)

D_einsum = pairwise_dist_einsum(X_large)
print(f"Einsum matches loop: {np.allclose(D_loop, D_einsum, atol=1e-10)}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Pairwise distance heatmap between samples
import matplotlib.pyplot as plt

X_small = rng.exponential(5.0, size=(200, 8))
D = pairwise_dist_einsum(X_small)
labels = [f"S{i+1}" for i in range(8)]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(D, cmap="viridis")
ax.set_xticks(range(8)); ax.set_xticklabels(labels)
ax.set_yticks(range(8)); ax.set_yticklabels(labels)
plt.colorbar(im, ax=ax, label="Euclidean distance")
ax.set_title("Sample-sample distance matrix")
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `cpm_normalize(X)` in `compbio_utils/stats.py`. It should raise `ValueError`
   if any column sum is zero.
2. Without running the code, predict the output shape of:
   - `np.ones((3,4,5)) + np.ones((4,5))`
   - `np.ones((3,4,5)) + np.ones((3,1,5))`
   - `np.ones((3,4,5)) + np.ones((5,))`
   Then verify by running.
3. Rewrite `sliding_gc` from NB03 using NumPy vectorization instead of a list comprehension.
   Benchmark the speedup on a 100 kb synthetic sequence.

---
## Quiz — Active Recall

1. What is the broadcasting rule? State it precisely.
2. A matrix is shape `(5000, 20)`. Its per-sample means are shape `(20,)`. Write the
   single NumPy line that subtracts per-sample means from every gene.
3. Why is the vectorized pairwise distance faster than the nested loop?
4. What does `np.newaxis` do? Give an example.
5. Why does CPM normalization fail if a column sum is zero?

---
## Reflection

**Date completed:** ____________________

> *[Could you draw the broadcasting alignment on paper for each example in Cell 6.2? Was the speedup surprising? Is cpm_normalize in compbio_utils/stats.py with tests?]*

---
*Next: `06_numpy_linear_algebra.ipynb`*